# Day 5 · Senses: Sight — ท้องฟ้าเปิดแค่ไหนจากจุดดูดาว 24 แห่ง

**#30DayMapChallenge 2026 · วันที่ 5 พ.ย. · ธีม Senses: Sight** — "How do you see something? Maybe map a viewshed?"

ต่อจาก Day 4 (24 hotspot ฟ้ามืด) — วันนี้เอา **หอดูดาวใกล้สุดประจำแต่ละกลุ่ม** เป็นจุดสังเกตการณ์
แล้วคำนวณ **viewshed จาก DEM** สองแบบ: (B) **sky-viewshed / เส้นขอบฟ้า** ว่าฟ้าเปิดแค่ไหน และ
(A) **terrain viewshed** ว่ามองพื้นเห็นบริเวณใด

> **รันตามได้เลย: Runtime → Run all** — โน้ตบุ๊กนี้อ่านผลลัพธ์ Day 4 (`day04_clusters.geojson`) และ
> `darksky_thailand.csv` จาก GitHub ไม่ต้องคำนวณใหม่ ถ้าโหลดไม่ได้จะมีปุ่มอัปโหลดไฟล์เอง

In [ ]:
# ติดตั้ง library (numpy/pandas/matplotlib มีใน Colab อยู่แล้ว)
!pip install -q rasterio
import rasterio
print('rasterio', rasterio.__version__)

## 0. เตรียมฟอนต์ไทย

ใช้ **Noto Sans Thai** ตัวเดียวกับ Day 1–4 (วาดสระ/วรรณยุกต์ซ้อนถูกต้อง)

In [ ]:
import urllib.request
from fontTools.ttLib import TTFont
from fontTools.varLib.instancer import instantiateVariableFont
from matplotlib import font_manager

urllib.request.urlretrieve(
    'https://github.com/google/fonts/raw/main/ofl/notosansthai/NotoSansThai%5Bwdth%2Cwght%5D.ttf',
    'NotoSansThai-VF.ttf')
for _name,_w in [('NotoSansThai-Regular',400),('NotoSansThai-Bold',700)]:
    _f=TTFont('NotoSansThai-VF.ttf'); instantiateVariableFont(_f,{'wght':_w,'wdth':100},inplace=True)
    _f.save(f'{_name}.ttf'); font_manager.fontManager.addfont(f'{_name}.ttf')
import matplotlib.pyplot as plt
plt.rcParams['font.family']='Noto Sans Thai'
print('ติดตั้งฟอนต์ไทยเรียบร้อย')

## 1. โหลดผลลัพธ์ Day 4 → จุดสังเกตการณ์ 24 แห่ง

แต่ละ cluster มีคอลัมน์ `nearest` = ชื่อหอดูดาว NARIT ที่ใกล้สุด ผมหาชื่อที่ * dominating * ของแต่ละ rank
แล้ว map กลับเป็น lat/lon จาก CSV

In [ ]:
import json, numpy as np, pandas as pd

BASE='https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/'
def fetch(rel):
    try:
        return json.load(__import__('urllib.request',fromlist=['urlopen']).urlopen(BASE+rel,timeout=60))
    except Exception:
        from google.colab import files
        up=files.upload(); return json.loads(next(iter(up.values())).decode('utf-8'))

g4=fetch('posts/day04-clusters/day04_clusters.geojson')

# CSV พิกัดหอดูดาว
try:
    df=pd.read_csv(BASE+'data/darksky_thailand.csv')
except Exception:
    from google.colab import files
    up=files.upload(); df=pd.read_csv(__import__('io').StringIO(next(iter(up.values())).decode('utf-8')))

from collections import Counter,defaultdict
byrank=defaultdict(Counter)
for f in g4['features']:
    byrank[f['properties']['rank']][f['properties']['nearest']]+=1
lut={str(n).strip():(la,lo) for n,la,lo in zip(df['name'],df['latitude'],df['longitude'])}
sites=[]
for r in sorted(byrank):
    nm=byrank[r].most_common(1)[0][0]
    la,lo=lut[str(nm).strip()]
    sites.append(dict(rank=r,name=nm,lat=la,lon=lo))
print('observer points:',len(sites))
for s in sites[:5]: print(' #%d'%s['rank'],round(s['lat'],3),round(s['lon'],3),s['name'][:24])

## 2. DEM จาก terrarium tiles (ฟรี ไม่ต้อง login)

terrarium เข้ารหัสความสูงไว้ใน RGB: `h = R·256 + G + B/256 − 32768` (เมตร) ที่ zoom 13 ได้ ~18 ม./พิกเซล
เราประกอบ tile 3×3 เป็นหน้าต่าง ~±7 กม. รอบจุด ซึ่งรัศมีพอให้เส้นขอบฟ้า stabilise

In [ ]:
import math, urllib.request
from rasterio.io import MemoryFile
Z=13; SPAN=1; _cache={}

def tile(xt,yt):
    key=(Z,xt,yt)
    if key in _cache: return _cache[key]
    h=np.full((256,256),np.nan,float)
    url=f'https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{Z}/{xt}/{yt}.png'
    try:
        data=urllib.request.urlopen(url,timeout=40).read()
        with MemoryFile(data) as mem:
            with mem.open() as ds: rgb=ds.read()
        h=rgb[0].astype('float64')*256+rgb[1]+rgb[2]/256.0-32768.0
    except Exception as e:
        print('tile fail',e)
    _cache[key]=h; return h

def window(lat,lon,span=SPAN):
    n=2**Z
    gxf=(lon+180)/360*n; gyf=(1-math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n
    xt0,yt0=int(gxf),int(gyf); size=2*span+1
    BIG=np.full((size*256,size*256),np.nan,float)
    for dx in range(-span,span+1):
        for dy in range(-span,span+1):
            BIG[(dy+span)*256:(dy+span+1)*256,(dx+span)*256:(dx+span+1)*256]=tile(xt0+dx,yt0+dy)
    res_m=156543.0339*math.cos(math.radians(lat))/n
    opx=int(round((gxf-(xt0-span))*256)); opy=int(round((gyf-(yt0-span))*256))
    return BIG,res_m,opx,opy
print('terrarium helper พร้อม')

## 3. B) Sky-viewshed — เส้นขอบฟ้า 360° และค่า open-sky

เดินรังสีครบ 360 ทิศ บันทึกมุมเงยสูงสุดของ terrain (ขอบฟ้า) แล้ว `open_sky =` เฉลี่ยของมุมที่ว่างเหนือขอบฟ้า / 90°

In [ ]:
def sky_viewshed(dem,res_m,opx,opy,eye=2.0):
    H,W=dem.shape; horizon=np.zeros(360)
    opy=min(max(opy,0),H-1); opx=min(max(opx,0),W-1)
    z_obs=float(np.nanmean(dem[max(0,opy-1):opy+2,max(0,opx-1):opx+2]))+eye
    maxrange=int(min(H,W)/2)-1
    for a in range(360):
        th=math.radians(a); cx,cy=math.sin(th),math.cos(th); hmax=-90.0; r=1.0
        while r<maxrange:
            x=int(round(opx+cx*r)); y=int(round(opy-cy*r))
            if 0<=x<W and 0<=y<H:
                z=dem[y,x]
                if not np.isnan(z):
                    ang=math.degrees(math.atan2(z-z_obs,r*res_m))
                    if ang>hmax: hmax=ang
            r+=1.0
        horizon[a]=hmax
    open_sky=float(np.mean((90-np.clip(horizon,0,90))/90))
    return horizon,z_obs,open_sky

rows=[]
for s in sites:
    dem,res_m,opx,opy=window(s['lat'],s['lon'])
    horizon,z_obs,osk=sky_viewshed(dem,res_m,opx,opy)
    s.update(z_obs=z_obs,open_sky=osk,mean_horizon=float(np.clip(horizon,0,90).mean()),max_horizon=float(horizon.max()))
    np.save('hor_%d.npy'%s['rank'],horizon)
    rows.append(s)
    print('#%2d open_sky=%.3f meanH=%5.1f maxH=%5.1f z=%5.0fm  %s'%(s['rank'],osk,np.clip(horizon,0,90).mean(),horizon.max(),z_obs,s['name'][:24]))
print('\nmean open_sky=%.3f'%np.mean([s['open_sky'] for s in rows]))

## 4. แผนที่หลัก (B) — open-sky ของ 24 จุด ทับบนหกเหลี่ยม Day 3

In [ ]:
import matplotlib; matplotlib.rcParams['font.family']='Noto Sans Thai'
from matplotlib.colors import LinearSegmentedColormap,Normalize
from matplotlib.collections import PolyCollection
DARK_BG='#0b1020'; SKY=LinearSegmentedColormap.from_list('sky',['#f43f5e','#facc15','#22d3ee']); NORM=Normalize(0.80,1.00)

g3=fetch('posts/day03-polygons/day03_hex_polygons.geojson')
def rings_of(g): return [g['coordinates'][0]] if g['type']=='Polygon' else [p[0] for p in g['coordinates']]
backdrop=[np.array(r) for f in g3['features'] for r in rings_of(f['geometry'])]
bx=np.concatenate([c[:,0] for c in backdrop]); by=np.concatenate([c[:,1] for c in backdrop])

fig,ax=plt.subplots(figsize=(10,11.5),facecolor=DARK_BG); ax.set_facecolor(DARK_BG)
ax.add_collection(PolyCollection(backdrop,facecolor='#141a30',edgecolor='#222842',lw=0.2))
sc=ax.scatter([s['lon'] for s in rows],[s['lat'] for s in rows],s=210,c=[s['open_sky'] for s in rows],cmap=SKY,norm=NORM,edgecolors='white',linewidths=0.8,zorder=5)
for s in rows: ax.text(s['lon'],s['lat'],str(s['rank']),color='#0b1020',fontsize=8,fontweight='bold',ha='center',va='center',zorder=6)
ax.set_xlim(bx.min()-0.3,bx.max()+0.3); ax.set_ylim(by.min()-0.3,by.max()+0.3); ax.set_aspect('equal')
ax.set_title('Day 5 · Senses: Sight — ท้องฟ้าเปิดแค่ไหนจากจุดดูดาว 24 แห่ง',color='white',fontsize=14,pad=12,fontweight='bold')
cb=fig.colorbar(sc,ax=ax,fraction=0.04,pad=0.02); cb.set_label('สัดส่วนฟ้าเปิด (open-sky)',color='#cdd6ff'); cb.ax.tick_params(colors='#8a94b8')
fig.text(0.5,0.012,'ยิ่งแดง = ภูเขารอบข้างบังขอบฟ้ามาก · ยิ่งฟ้า = ที่ราบ/ชายทะเลเปิดโล่ง | DEM terrarium ~18m',color='#8a94b8',fontsize=8.5,ha='center')
fig.savefig('day05_hero.png',dpi=160,facecolor=DARK_BG,bbox_inches='tight'); plt.close(fig); print('saved day05_hero.png')

## 5. กุหลาบขอบฟ้า (6 จุด: ถูกบังมากสุด 3 + ฟ้าเปิดสุด 3)

In [ ]:
blocked=sorted(rows,key=lambda s:s['open_sky'])[:3]; openest=sorted(rows,key=lambda s:-s['open_sky'])[:3]; picks=blocked+openest
fig,axes=plt.subplots(2,3,figsize=(12,8),facecolor=DARK_BG,subplot_kw=dict(projection='polar'))
for ax,s in zip(axes.ravel(),picks):
    h=np.load('hor_%d.npy'%s['rank']); hh=np.r_[h,h[0]]
    ax.plot(np.radians(hh),np.clip(hh,0,40),color=SKY(NORM(s['open_sky'])),lw=1.8)
    ax.fill(np.radians(hh),np.clip(hh,0,40),color=SKY(NORM(s['open_sky'])),alpha=0.3)
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1); ax.set_ylim(0,40)
    ax.set_title('#%d %s\nopen-sky %.3f · max %.0f°'%(s['rank'],s['name'][:16],s['open_sky'],s['max_horizon']),color='white',fontsize=9.5,pad=14)
    ax.set_facecolor(DARK_BG); ax.grid(color='#3a466e',lw=0.5); ax.tick_params(colors='#8a94b8',labelsize=7)
    if s['open_sky']>0.99: ax.text(0,22,'ขอบฟ้าราบ\n~0°',ha='center',va='center',color='#8a94b8',fontsize=9)
fig.suptitle('เส้นขอบฟ้า (มุมที่ภูเขาบัง) — ซ้าย=ถูกบังมาก, ขวา=ฟ้าเปิด',color='white',fontsize=13,fontweight='bold')
fig.savefig('day05_roses.png',dpi=150,facecolor=DARK_BG,bbox_inches='tight'); plt.close(fig); print('saved day05_roses.png')

## 6. A) Terrain viewshed จาก 3 ยอดเขาสูง

ยิงรังสีละเอียด 1,440 ทิศ (ทุก 0.25°) แบบ sub-pixel เพื่อให้ fan ต่อเนื่อง ไม่ใช่จุดกระจาย

In [ ]:
def dense_viewshed(dem,res_m,opx,opy,eye=15.0,naz=1440,step=0.5):
    H,W=dem.shape; vis=np.zeros((H,W),bool)
    opy=min(max(opy,0),H-1); opx=min(max(opx,0),W-1)
    z_obs=float(np.nanmean(dem[opy-1:opy+2,opx-1:opx+2]))+eye
    maxr=min(H,W)/2-1
    for a in range(naz):
        th=2*math.pi*a/naz; cx,cy=math.sin(th),math.cos(th); hmax=-90.0; r=1.0
        while r<maxr:
            x=int(round(opx+cx*r)); y=int(round(opy-cy*r))
            if 0<=x<W and 0<=y<H:
                z=dem[y,x]
                if not np.isnan(z):
                    ang=math.degrees(math.atan2(z-z_obs,r*res_m))
                    vis[y,x]=ang>=hmax-1e-3
                    if ang>hmax: hmax=ang
            r+=step
    return vis,z_obs

S={s['rank']:s for s in rows}; fan_ranks=[2,14,22]
fig,axes=plt.subplots(1,3,figsize=(15,5.6),facecolor=DARK_BG)
for ax,r in zip(axes,fan_ranks):
    s=S[r]; dem,res_m,opx,opy=window(s['lat'],s['lon'],span=2); vis,z_obs=dense_viewshed(dem,res_m,opx,opy)
    rad_km=(min(dem.shape)/2)*res_m/1000
    ax.imshow(dem,cmap='terrain',origin='upper')
    ax.imshow(np.ma.masked_where(~vis,vis.astype(float)),cmap=LinearSegmentedColormap.from_list('v',['#22d3ee','#22d3ee']),alpha=0.72,origin='upper')
    ax.plot(opx,opy,marker='*',ms=15,color='white',mec='red',mew=1.2)
    ax.set_title('#%d %s\nระดับสายตา %.0f ม. · รัศมี ~%.0f กม.'%(r,s['name'][:18],z_obs,rad_km),color='white',fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('A) Terrain viewshed จาก 3 ยอดเขาสูง (cyan = พื้นที่ที่สายตาสัมผัสได้) · DEM ~18 ม. · eye 15 ม.',color='white',fontsize=13,fontweight='bold')
fig.savefig('day05_fans.png',dpi=150,facecolor=DARK_BG,bbox_inches='tight'); plt.close(fig); print('saved day05_fans.png')

## 7. ส่งออก geojson (24 จุด + ค่า viewshed)

In [ ]:
feats=[{'type':'Feature','geometry':{'type':'Point','coordinates':[s['lon'],s['lat']]},
        'properties':{'rank':s['rank'],'name':s['name'],'open_sky':round(s['open_sky'],3),
                      'mean_horizon':round(s['mean_horizon'],1),'max_horizon':round(s['max_horizon'],1),
                      'z_obs':round(s['z_obs'])}} for s in rows]
gj={'type':'FeatureCollection','name':'day05_sight','features':feats}
json.dump(gj,open('day05_sight.geojson','w',encoding='utf-8'),ensure_ascii=False)
from google.colab import files
files.download('day05_sight.geojson')
print('เขียน day05_sight.geojson (%d จุด)'%len(feats))

---

## อัปเดต 7 พ.ย. 2569 — รูปทั้งสามถูกวาดใหม่ (restamp)

โค้ดข้างบนคือเวอร์ชันวันนั้น ซึ่งยัง (ก) ใช้เลขกลุ่ม Day 4 เป็นเลขใหญ่บนแผนที่ และ
(ข) พล็อตวงเส้นขอบฟ้าด้วย `theta = radians(มุมที่ภูเขาบัง)` เส้นจึงถูกวาดแค่ช่วง 0–45°
ไม่ใช่ 360 ทิศตามที่หัวข้อบอก — ทั้งสองอย่างแก้แล้วในเซตเซลล์ด้านล่าง
(บริบทเพิ่มเติมอยู่ในหัวข้อ "ว่าด้วยตัวเลขบนรูป" ของโพสต์)

รันได้ทั้งบนเครื่องผู้เขียนและบน Colab · บน Colab จะดึง geojson จาก GitHub เอง


### ตั้งค่า: พาเลท, โฟลเดอร์, shim Colab, ฟอนต์ Noto Sans Thai

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 5 restamp — regenerate the three figures of posts/day05-sight with correct numbering.

Rule adopted (see Day 6 post): the big number on a map must be the rank of the quantity that
map actually shows. Day 5 shows open-sky, so its big number must be the open-sky rank; the
Day 4 size-rank moves to a small "#NN" tag (matching the symbol Day 4 itself prints).

Visual style is copied from notebooks/day05_sight.ipynb so the figures still look like Day 5.
"""
import io, os, json, math, time, urllib.request
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colorbar import ColorbarBase

# รันได้ทั้งบนเครื่องตัวเองและบน Colab
COLAB = not os.path.isdir(r"D:\GIS30Days\geo-notes")
WORK  = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO  = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "day05_cache") if COLAB else os.path.join(WORK, "cache", "day06")
OUT   = os.path.join(WORK, "out", "day05")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
if COLAB:
    RAW = "https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/"
    for rel in ["posts/day05-sight/day05_sight.geojson", "posts/day03-polygons/day03_hex_polygons.geojson"]:
        dst = os.path.join(CACHE, os.path.basename(rel))
        if not os.path.exists(dst): urllib.request.urlretrieve(RAW + rel, dst)
    REPO = CACHE          # ไฟล์ทั้งสองถูกดึงมาไว้ใน CACHE แล้ว
    def _flat(rel):       # ให้ path แบบ "posts/..." ใช้ basename ได้
        return os.path.basename(rel)
    print("Colab: ดึง geojson เรียบร้อย")

for f in sorted(os.listdir(CACHE)):
    if f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, f))

def have(name):
    return any(e.name == name for e in font_manager.fontManager.ttflist)

def noto_sans_thai():
    """รูป Day 5 ของเดิมใช้ Noto Sans Thai — ดึงแบบเดียวกับ Day 1 (variable + instancer)"""
    try:
        from fontTools.ttLib import TTFont
        from fontTools.varLib.instancer import instantiateVariableFont
        raw = os.path.join(CACHE, "NotoSansThai-VF.ttf")
        if not os.path.exists(raw):
            urllib.request.urlretrieve(
                "https://github.com/google/fonts/raw/main/ofl/notosansthai/"
                "NotoSansThai%5Bwdth%2Cwght%5D.ttf", raw)
        for nm, w in (("NotoSansThai-Regular", 400), ("NotoSansThai-Bold", 700)):
            out = os.path.join(CACHE, nm + ".ttf")
            if not os.path.exists(out):
                f = TTFont(raw); instantiateVariableFont(f, {"wght": w, "wdth": 100}, inplace=True); f.save(out)
            font_manager.fontManager.addfont(out)
        return True
    except Exception as e:
        print("    (noto fail:", type(e).__name__, e, ")")
        return False

if not have("Noto Sans Thai"):
    noto_sans_thai()
plt.rcParams["font.family"] = "Noto Sans Thai" if have("Noto Sans Thai") else "Trirong"
print("[font]", plt.rcParams["font.family"])

DARK_BG = "#0b1020"
SKY = LinearSegmentedColormap.from_list("sky", ["#f43f5e", "#facc15", "#22d3ee"])
NORM = Normalize(0.80, 1.00)

### data

In [ ]:
g5 = json.load(open(os.path.join(REPO, "day05_sight.geojson" if COLAB else "posts/day05-sight/day05_sight.geojson"), encoding="utf-8"))
g3 = json.load(open(os.path.join(REPO, "day03_hex_polygons.geojson" if COLAB else "posts/day03-polygons/day03_hex_polygons.geojson"), encoding="utf-8"))
obs = [dict(rank=int(f["properties"]["rank"]), name=f["properties"]["name"],
            lon=f["geometry"]["coordinates"][0], lat=f["geometry"]["coordinates"][1],
            open_sky=float(f["properties"]["open_sky"]), mean_horizon=float(f["properties"]["mean_horizon"]),
            max_horizon=float(f["properties"]["max_horizon"]), z_obs=float(f["properties"]["z_obs"]))
       for f in g5["features"]]
for i, o in enumerate(sorted(obs, key=lambda z: -z["open_sky"]), 1):
    o["open_rank"] = i
S = {o["rank"]: o for o in obs}
print(f"[data] {len(obs)} จุด · ฟ้าเปิดอันดับ 1 = {min(obs, key=lambda z: z['open_rank'])['name']}")

### terrarium helpers (same as Day 5/6)

In [ ]:
Z = 13
def tile(z, x, y, tries=3):
    url = f"https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png"
    for k in range(tries):
        try:
            rgb = np.asarray(Image.open(io.BytesIO(urllib.request.urlopen(url, timeout=40).read())).convert("RGB"), np.float32)
            return rgb[:, :, 0] * 256 + rgb[:, :, 1] + rgb[:, :, 2] / 256.0 - 32768.0
        except Exception:
            if k == tries - 1:
                return np.full((256, 256), np.nan, float)

def window(lat, lon, span=1):
    n = 2 ** Z
    gxf = (lon + 180) / 360 * n; gyf = (1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n
    xt0, yt0 = int(gxf), int(gyf); size = 2 * span + 1
    BIG = np.full((size * 256, size * 256), np.nan, float)
    for dx in range(-span, span + 1):
        for dy in range(-span, span + 1):
            BIG[(dy + span) * 256:(dy + span + 1) * 256, (dx + span) * 256:(dx + span + 1) * 256] = tile(Z, xt0 + dx, yt0 + dy)
    res = 156543.0339 * math.cos(math.radians(lat)) / n
    return BIG, res, int(round((gxf - (xt0 - span)) * 256)), int(round((gyf - (yt0 - span)) * 256))

def sky_viewshed(dem, res_m, opx, opy, eye=2.0):
    Hh, Wd = dem.shape; hz = np.zeros(360)
    opy = min(max(opy, 0), Hh - 1); opx = min(max(opx, 0), Wd - 1)
    z = float(np.nanmean(dem[max(0, opy - 1):opy + 2, max(0, opx - 1):opx + 2])) + eye
    maxr = int(min(Hh, Wd) / 2) - 1
    for a in range(360):
        th = math.radians(a); cx, cy = math.sin(th), math.cos(th); hm = -90.0; r = 1.0
        while r < maxr:
            xx = int(round(opx + cx * r)); yy = int(round(opy - cy * r))
            if 0 <= xx < Wd and 0 <= yy < Hh:
                v = dem[yy, xx]
                if not np.isnan(v):
                    ang = math.degrees(math.atan2(v - z, r * res_m))
                    if ang > hm: hm = ang
            r += 1.0
        hz[a] = hm
    return hz

def dense_viewshed(dem, res_m, opx, opy, eye=15.0, naz=1440, step=0.5):
    Hh, Wd = dem.shape; vis = np.zeros((Hh, Wd), bool)
    opy = min(max(opy, 0), Hh - 1); opx = min(max(opx, 0), Wd - 1)
    z = float(np.nanmean(dem[opy - 1:opy + 2, opx - 1:opx + 2])) + eye
    maxr = min(Hh, Wd) / 2 - 1
    for a in range(naz):
        th = 2 * math.pi * a / naz; cx, cy = math.sin(th), math.cos(th); hmax = -90.0; r = 1.0
        while r < maxr:
            x = int(round(opx + cx * r)); y = int(round(opy - cy * r))
            if 0 <= x < Wd and 0 <= y < Hh:
                v = dem[y, x]
                if not np.isnan(v):
                    ang = math.degrees(math.atan2(v - z, r * res_m))
                    vis[y, x] = ang >= hmax - 1e-3
                    if ang > hmax: hmax = ang
            r += step
    return vis, z

### 1. hero map

In [ ]:
print("[1] hero map ...")
def rings_of(g):
    return [g["coordinates"][0]] if g["type"] == "Polygon" else [p[0] for p in g["coordinates"]]
backdrop = [np.array(r) for f in g3["features"] for r in rings_of(f["geometry"])]
bx = np.concatenate([c[:, 0] for c in backdrop]); by = np.concatenate([c[:, 1] for c in backdrop])

fig, ax = plt.subplots(figsize=(10, 11.5), facecolor=DARK_BG); ax.set_facecolor(DARK_BG)
ax.add_collection(PolyCollection(backdrop, facecolor="#141a30", edgecolor="#222842", lw=0.2))
sc = ax.scatter([s["lon"] for s in obs], [s["lat"] for s in obs],
                s=210, c=[s["open_sky"] for s in obs], cmap=SKY, norm=NORM,
                edgecolors="white", linewidths=0.8, zorder=5)
for s in obs:
    ax.text(s["lon"], s["lat"], str(s["open_rank"]), color="#0b1020", fontsize=8.2,
            fontweight="bold", ha="center", va="center", zorder=6)
    ax.text(s["lon"] + 0.10, s["lat"] - 0.10, f"#{s['rank']}", color="#cbd5e1", fontsize=5.6,
            ha="left", va="top", zorder=6)
ax.set_xlim(bx.min() - 0.3, bx.max() + 0.3); ax.set_ylim(by.min() - 0.3, by.max() + 0.3)
ax.set_aspect("equal")
ax.set_title("Day 5 · Senses: Sight — ท้องฟ้าเปิดแค่ไหนจากจุดดูดาว 24 แห่ง",
             color="white", fontsize=14, pad=34, fontweight="bold")
ax.text(0.5, 1.052, "เลขใหญ่ = อันดับความฟ้าเปิด (open-sky) · #NN = อันดับขนาดกลุ่มจาก Day 4",
        transform=ax.transAxes, ha="center", va="bottom", color="#94a3b8", fontsize=9)
cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("open-sky (1 = เห็นโดมฟ้าครบ)", color="white"); cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
fig.text(0.01, 0.01, "ข้อมูล: NARIT darksky · DEM terrarium (Copernicus GLO-30) · sky-viewshed 360 ทิศ",
         color="#64748b", fontsize=7.5)
p = os.path.join(OUT, "day05_sight_preview.png")
fig.savefig(p, dpi=160, facecolor=DARK_BG, bbox_inches="tight"); plt.close(fig)
print("   ", p, f"{os.path.getsize(p)/1e6:.2f} MB")

### 2. roses (2x3 of the 6 picks)

In [ ]:
print("[2] roses panel ...")
hz_npy = os.path.join(CACHE, "horizons24.npz")
if os.path.exists(hz_npy):
    HZ = np.load(hz_npy)["H"]
    chk = np.clip(HZ, 0, 90).mean(axis=1)
    ref = np.array([o["mean_horizon"] for o in obs])
    assert np.abs(chk - ref).max() < 0.1, "horizon cache order does not match day05_sight.geojson"
    print("    ใช้ horizon cache (ยืนยันแล้วว่าเรียงตรงกับ geojson)")
else:
    t0 = time.time(); HZ = np.zeros((len(obs), 360))
    for i, s in enumerate(obs):
        dem, res, ox, oy = window(s["lat"], s["lon"])
        HZ[i] = sky_viewshed(dem, res, ox, oy)
    np.savez_compressed(hz_npy, H=HZ); print(f"    คำนวณใหม่ {time.time()-t0:.0f} วิ")
blocked = sorted(obs, key=lambda s: s["open_sky"])[:3]
openest = sorted(obs, key=lambda s: -s["open_sky"])[:3]
picks = blocked + openest
fig, axes = plt.subplots(2, 3, figsize=(12, 8), facecolor=DARK_BG, subplot_kw=dict(projection="polar"))
for ax, s in zip(axes.ravel(), picks):
    h = HZ[[o["rank"] for o in obs].index(s["rank"])]
    hh = np.r_[h, h[0]]
    az = np.radians(np.arange(361) % 360)          # azimuth ไม่ใช่ค่ามุมขอบฟ้า (บั๊กของรูปเดิม)
    rad = np.clip(hh, 0, 40)
    ax.plot(az, rad, color=SKY(NORM(s["open_sky"])), lw=1.8)
    ax.fill(az, rad, color=SKY(NORM(s["open_sky"])), alpha=0.35)
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.set_ylim(0, 40)
    ax.set_title(f"อันดับ {s['open_rank']} · Day 4 #{s['rank']}  {s['name'][:18]}\n"
                 f"open-sky {s['open_sky']:.3f} · max {s['max_horizon']:.0f}°",
                 color="white", fontsize=9.2, pad=18)
    ax.set_facecolor(DARK_BG); ax.grid(color="#3a466e", lw=0.5)
    ax.tick_params(colors="#8a94b8", labelsize=7)
    if s["open_sky"] > 0.99:
        ax.text(np.radians(200), 20, "ขอบฟ้าราบ" + chr(10) + "~0°", ha="center", va="center", color="#8a94b8", fontsize=9)
fig.suptitle("เส้นขอบฟ้า 360 ทิศ (มุมที่ยอดเขาด้านนั้นบังสายตา) — บน=ถูกบังมาก, ล่าง=ฟ้าเปิด",
             color="white", fontsize=13, fontweight="bold", y=0.995)
fig.subplots_adjust(top=0.86, bottom=0.06, hspace=0.55, wspace=0.25)
p2 = os.path.join(OUT, "day05_roses.png")
fig.savefig(p2, dpi=150, facecolor=DARK_BG, bbox_inches="tight"); plt.close(fig)
print("   ", p2, f"{os.path.getsize(p2)/1e6:.2f} MB")

### 3. fans (terrain viewshed, 3 mountains)

In [ ]:
print("[3] fans panel (dense_viewshed 3 จุด) ...")
VIS = os.path.join(CACHE, "fans_vis.npz")
fan_ranks = [2, 14, 22]
if os.path.exists(VIS):
    d = np.load(VIS); fans = [(d[f"v{i}"], float(d[f"z{i}"]), float(d[f"rad{i}"]), d[f"dem{i}"]) for i in range(3)]
    print("    ใช้ cache เดิม")
else:
    t0 = time.time(); fans = []
    for r in fan_ranks:
        s = S[r]
        dem, res_m, opx, opy = window(s["lat"], s["lon"], span=2)
        vis, z_obs = dense_viewshed(dem, res_m, opx, opy)
        rad_km = (min(dem.shape) / 2) * res_m / 1000
        fans.append((vis, z_obs, rad_km, dem))
        print(f"    c{r} {s['name'][:22]:22} z={z_obs:.0f} ม. รัศมี {rad_km:.1f} กม. ({time.time()-t0:.0f} วิ)")
    np.savez_compressed(VIS, **{f"v{i}": v for i, (v, z, rk, dm) in enumerate(fans)},
                        **{f"z{i}": z for i, (v, z, rk, dm) in enumerate(fans)},
                        **{f"rad{i}": rk for i, (v, z, rk, dm) in enumerate(fans)},
                        **{f"dem{i}": dm for i, (v, z, rk, dm) in enumerate(fans)})
fig, axes = plt.subplots(1, 3, figsize=(15, 5.6), facecolor=DARK_BG)
VC = LinearSegmentedColormap.from_list("v", ["#22d3ee", "#22d3ee"])
for ax, r, (vis, z_obs, rad_km, dem) in zip(axes, fan_ranks, fans):
    s = S[r]
    ax.imshow(dem, cmap="terrain", origin="upper")
    ax.imshow(np.ma.masked_where(~vis, vis.astype(float)), cmap=VC, alpha=0.72, origin="upper")
    opx, opy = 2 * 256, 2 * 256
    ax.plot(opx, opy, marker="*", ms=15, color="white", mec="red", mew=1.2)
    ax.set_title(f"Day 4 #{r} · อันดับฟ้าเปิด {s['open_rank']}  {s['name'][:18]}\n"
                 f"ระดับสายตา {z_obs:.0f} ม. · รัศมี ~{rad_km:.0f} กม.", color="white", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("A) Terrain viewshed จาก 3 ยอดเขาสูง (cyan = พื้นที่ที่สายตาสัมผัสได้) · DEM ~18 ม. · eye 15 ม.",
             color="white", fontsize=13, fontweight="bold")
p3 = os.path.join(OUT, "day05_fans.png")
fig.savefig(p3, dpi=150, facecolor=DARK_BG, bbox_inches="tight"); plt.close(fig)
print("   ", p3, f"{os.path.getsize(p3)/1e6:.2f} MB")

### 4. verification

In [ ]:
print("[4] ตรวจ label ↔ ข้อมูล (ทุกตัวเลขที่วาดต้อง map กลับ geojson ได้) ...")
lut = {o["rank"]: o for o in obs}
rows = []
for o in sorted(obs, key=lambda z: z["open_rank"]):
    rows.append((o["open_rank"], o["rank"], o["name"], round(o["open_sky"], 3), round(o["max_horizon"], 1)))
assert len({r[0] for r in rows}) == 24 and sorted(r[0] for r in rows) == list(range(1, 25))
assert len({r[1] for r in rows}) == 24
for r in rows[:3] + rows[-2:]:
    print(f"    อันดับ {r[0]:2d} = #{r[1]:2d}  open-sky {r[3]:.3f} · max {r[4]:.1f}°  {r[2][:30]}")
print("    OK: อันดับ 1–24 ไม่ซ้ำกัน, #NN ไม่ซ้ำกัน, ตรงกับไฟล์ข้อมูล")